In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

1.. Finding the Total Number of Ratings in the Dataset 

In [2]:
ratings=pd.read_csv("ratings.csv")
ratings

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [3]:
rows,columns=ratings.shape
print("Total number of ratings:",rows)

Total number of ratings: 100836


2. Identifying the Movie with the Highest Average Rating 
(with at least 50 ratings)

In [4]:
movies=pd.read_csv("movies.csv")
movies

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation


In [5]:
grouped=ratings.groupby('movieId')
rating_count=grouped['rating'].count()
print("Count of rating for each movie is:",rating_count)

Count of rating for each movie is: movieId
1         215
2         110
3          52
4           7
5          49
         ... 
193581      1
193583      1
193585      1
193587      1
193609      1
Name: rating, Length: 9724, dtype: int64


In [6]:
average=grouped['rating'].mean()
print("Average rating for each movie is:",average)

Average rating for each movie is: movieId
1         3.920930
2         3.431818
3         3.259615
4         2.357143
5         3.071429
            ...   
193581    4.000000
193583    3.500000
193585    3.500000
193587    3.500000
193609    4.000000
Name: rating, Length: 9724, dtype: float64


In [7]:
filtered_data=grouped.filter(lambda x:len(x)>=50)
filtered_data

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100657,610,106782,4.5,1479542155
100672,610,109374,4.5,1493845137
100673,610,109487,3.5,1493845041
100701,610,112852,4.5,1493845402


In [8]:
maximum=average.max()
maximum

5.0

In [9]:
highest_id=average.idxmax()
highest_id

53

In [10]:
print(f'movie with id:{highest_id} has highest rating of {maximum}')

movie with id:53 has highest rating of 5.0


In [11]:
movies.columns

Index(['movieId', 'title', 'genres'], dtype='object')

In [12]:
movies.columns = movies.columns.str.strip() 
moviename = movies.loc[highest_id, 'title']
print('Movie with highest rating is:', moviename)


Movie with highest rating is: Indian in the Cupboard, The (1995)


3. Determining the Most Common Rating Given by Users

In [13]:
common=ratings['rating'].mode()
print("Most common rating:",common)

Most common rating: 0    4.0
Name: rating, dtype: float64


4. Retrieving the IMDb Rating of the Highest-Rated Movie 

In [14]:
links=pd.read_csv("links.csv", index_col=0)
links

,imdbId,tmdbId
movieId,,
1,114709,862.0
2,113497,8844.0
3,113228,15602.0
4,114885,31357.0
5,113041,11862.0
...,...,...
193581,5476944,432131.0
193583,5914996,445030.0
193585,6397426,479308.0


In [15]:
movies_reset = movies.reset_index()
merged_data = movies_reset.merge(links, on='movieId', how='inner')


In [16]:

imdb_id = merged_data.loc[merged_data['movieId'] == highest_id, 'imdbId'].values[0]

imdb_id = str(imdb_id).zfill(7)  


In [17]:
import requests
from bs4 import BeautifulSoup
imdb_url = f"https://www.imdb.com/title/tt{imdb_id}/"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(imdb_url, headers=headers)
if response.status_code == 200:
    soup = BeautifulSoup(response.text, "html.parser")
    rating_tag = soup.find("span", {"class": "sc-bde20123-1 cMEQkK"})  
    if not rating_tag:
        rating_tag = soup.find("span", {"class": "ipc-rating-star"})  
    if rating_tag:
        imdb_rating = rating_tag.text
    else:
        imdb_rating = "Not Found"

    print(f"IMDb Rating of the highest-rated movie: {imdb_rating}")

else:
    print(f"Failed to fetch IMDb page. Status code: {response.status_code}")


IMDb Rating of the highest-rated movie: 7.6


5. Counting Sci-Fi Movies with More Than 100 Ratings 

In [18]:
merged_df = pd.merge(ratings, movies, on="movieId")
merged_df

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller
...,...,...,...,...,...,...
100831,610,166534,4.0,1493848402,Split (2017),Drama|Horror|Thriller
100832,610,168248,5.0,1493850091,John Wick: Chapter Two (2017),Action|Crime|Thriller
100833,610,168250,5.0,1494273047,Get Out (2017),Horror
100834,610,168252,5.0,1493846352,Logan (2017),Action|Sci-Fi


In [19]:
sci_fi_movies = merged_df[merged_df["genres"].str.contains("Sci-Fi", na=False, case=False)]
sci_fi_counts = sci_fi_movies.groupby("movieId").size().reset_index(name="num_ratings")
popular_sci_fi = sci_fi_counts[sci_fi_counts["num_ratings"] > 100]
print(f"Number of Sci-Fi movies with more than 100 ratings: {len(popular_sci_fi)}")


Number of Sci-Fi movies with more than 100 ratings: 32


1. How many CSV files are in the dataset directory? 
• Check the dataset directory. 
• Count the number of .csv files using a script (e.g., os.listdir() in Python). 
• The answer depends on the dataset contents.

In [20]:
import os
dataset_directory ="D:/Internship/Task-4" 
all_files = os.listdir(dataset_directory)
csv_files = [file for file in all_files if file.endswith(".csv")]
print(f"Number of CSV files in the dataset directory: {len(csv_files)}")


Number of CSV files in the dataset directory: 4


2. What does movies_df.shape return? 
• Load the movies.csv file into a DataFrame. 
• Use .shape to get the tuple (rows, columns), which represents the number of rows 
(movies) and columns (attributes). 

In [21]:
movies.shape

(9742, 3)

3. What does ratings_df['userId'].nunique() return? 
• Load the ratings.csv file into a DataFrame. 
• Use .nunique() on the userId column to count the number of unique users.

In [22]:
ratings['userId'].nunique()

610

4. How is the most-rated movie identified in the dataset? 
• Load ratings.csv. 
• Count how many times each movieId appears using .value_counts(). 
• Identify the movie with the highest count.

In [23]:
most_rated_movie_id = ratings['movieId'].value_counts().idxmax()
print(f"The most-rated movie has movieId: {most_rated_movie_id}")

The most-rated movie has movieId: 356


5. What type of data does tags_df[tags_df['movieId'] == matrix_movie_id]['tag'].unique() 
return? 
• Load tags.csv. 
• Filter the DataFrame where movieId matches "The Matrix (1999)". 
• Extract unique tags using .unique(), which returns a list.

In [24]:
print(movies[movies['title'].str.contains("Matrix", case=False, na=False)])

filtered = movies[movies['title'].str.strip() == "The Matrix (1999)"]
if not filtered.empty:
    matrix_movie_id = filtered['movieId'].iloc[0]
    print("Movie ID for The Matrix (1999):", matrix_movie_id)
else:
    print("Movie ID for The Matrix (1999) not found.")

      movieId                           title  \
1939     2571              Matrix, The (1999)   
4351     6365     Matrix Reloaded, The (2003)   
4639     6934  Matrix Revolutions, The (2003)   
5669    27660           Animatrix, The (2003)   

                                     genres  
1939                 Action|Sci-Fi|Thriller  
4351  Action|Adventure|Sci-Fi|Thriller|IMAX  
4639  Action|Adventure|Sci-Fi|Thriller|IMAX  
5669          Action|Animation|Drama|Sci-Fi  
Movie ID for The Matrix (1999) not found.


6. If mean_rating = sum(ratings) / len(ratings), what does it calculate? 
• Load ratings.csv. 
• Compute the sum of all ratings. 
• Divide by the total number of ratings to get the average rating.

In [25]:
total=ratings['rating'].sum()
total

353083.0

In [26]:
length=len(ratings)

In [27]:
average_rating=total/length
average_rating

3.501556983616962

7. What does stats.skew(fight_club_ratings) measure? 
• Extract ratings for "Fight Club". 
• Use scipy.stats.skew() to measure the asymmetry of the rating distribution. 
• If skewness is positive, ratings are right-skewed; if negative, they are left-skewed.

In [28]:
fight_club_id = movies[movies["title"] == "Fight Club (1999)"]["movieId"].iloc[0]
fight_club_ratings = ratings[ratings["movieId"] == fight_club_id]["rating"]


In [29]:
from scipy.stats import skew
skewness = skew(fight_club_ratings)
print(f"Skewness of Fight Club ratings: {skewness}")

Skewness of Fight Club ratings: -1.8474937360359363


10. How is the Sci-Fi movie with the highest IMDb rating identified? 
• Load movies.csv and filter movies with the genre "Sci-Fi". 
• Use scrapper(imdbId) to get IMDb ratings. 
• Identify the highest-rated Sci-Fi movie.

In [30]:
import requests
from bs4 import BeautifulSoup
import numpy as np
import json
import time

def scrapper(imdbId):
    try:
        imdb_id_str = str(int(imdbId)).zfill(7)  
        URL = f"https://www.imdb.com/title/tt{imdb_id_str}/"
        print(f"Accessing URL: {URL}")  
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:119.0) Gecko/20100101 Firefox/119.0'
        }
        response = requests.get(URL, headers=headers)
        print(f"Response status code: {response.status_code}")  
        if response.status_code != 200:
            print("Failed to fetch IMDb page")
            return np.nan
        soup = BeautifulSoup(response.text, 'html.parser')
        script_tag = soup.find("script", type="application/ld+json")
        if script_tag:
            try:
                json_data = json.loads(script_tag.string)
                if "aggregateRating" in json_data and "ratingValue" in json_data["aggregateRating"]:
                    rating = json_data["aggregateRating"]["ratingValue"]
                    print(f"Found rating: {rating}")
                    return float(rating)
            except json.JSONDecodeError:
                print("Failed to parse JSON data")
        print("Rating not found")
        return np.nan

    except Exception as e:
        print(f"Error: {e}")
        return np.nan
    finally:
        time.sleep(2)  

In [31]:
movies = pd.read_csv("movies.csv")
links = pd.read_csv("links.csv")
sci_fi_movies = movies[movies["genres"].str.contains("Sci-Fi", na=False, case=False)]
sci_fi_movies = sci_fi_movies.merge(links, on="movieId")
sci_fi_movies["imdb_rating"] = sci_fi_movies["imdbId"].apply(scrapper)
highest_rated_movie = sci_fi_movies.loc[sci_fi_movies["imdb_rating"].idxmax()]
print(f"\n**Highest Rated Sci-Fi Movie:** {highest_rated_movie['title']}")
print(f" IMDb Rating: {highest_rated_movie['imdb_rating']}")
print(f"IMDb Link: https://www.imdb.com/title/tt{str(int(highest_rated_movie['imdbId'])).zfill(7)}/")


ERROR! Session/line number was not unique in database. History logging moved to new session 457
Accessing URL: https://www.imdb.com/title/tt0114168/
Response status code: 200
Found rating: 6.6
Accessing URL: https://www.imdb.com/title/tt0112682/
Response status code: 200
Found rating: 7.5
Accessing URL: https://www.imdb.com/title/tt0114746/
Response status code: 200
Found rating: 8
Accessing URL: https://www.imdb.com/title/tt0116839/
Response status code: 200
Found rating: 2.6
Accessing URL: https://www.imdb.com/title/tt0114367/
Response status code: 200
Found rating: 6.3
Accessing URL: https://www.imdb.com/title/tt0118040/
Response status code: 200
Found rating: 6
Accessing URL: https://www.imdb.com/title/tt0112715/
Response status code: 200
Found rating: 5.3
Accessing URL: https://www.imdb.com/title/tt0113481/
Response status code: 200
Found rating: 5.6
Accessing URL: https://www.imdb.com/title/tt0113492/
Response status code: 200
Found rating: 5.6
Accessing URL: https://www.imdb.com